In [1]:
path = "/content/drive/MyDrive/CERN/Dataset_Specific_labelled.h5"

In [2]:
import torch
import h5py
import random
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as T
import numpy as np

In [3]:
class H5Dataset(Dataset):
    def __init__(self, file_path, transform=None):
        self.file_path = file_path
        self.transform = transform

    def __getitem__(self, index):
        with h5py.File(self.file_path, 'r') as f:
            data = f['jet'][index]  # (H, W, 8)

        tensor = torch.from_numpy(data).float()  # (H, W, 8)
        tensor = tensor.permute(2, 0, 1)        # (8, H, W)

        if self.transform is not None:
            tensor = self.transform(tensor)

        return tensor

    def __len__(self):
        with h5py.File(self.file_path, 'r') as f:
            return len(f['jet'])

In [5]:
from torchvision import transforms
transform = transforms.Compose([
    transforms.Normalize(mean=[0.5] * 8, std=[0.5] * 8)
])

In [6]:
dataset = H5Dataset(path,transform=transform)

In [7]:
batch_size = 32
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=2)

In [8]:
import h5py
import numpy as np
import random

def extract_raw_h5_image(h5_path, dataset_name="jet"):
    # 1. Open the file in read-only mode
    with h5py.File(h5_path, 'r') as f:
        # Access the dataset (e.g., 'images' or 'X')
        dataset = f[dataset_name]
        total_images = dataset.shape[0]

        # 2. Select one random index
        random_idx = random.randint(0, total_images - 1)

        # 3. Pull the raw data (No normalization, no .transpose, no scaling)
        raw_image = dataset[random_idx]

        print(f"Extracted Image Index: {random_idx}")
        print(f"Raw Shape: {raw_image.shape}")
        print(f"Raw Data Type: {raw_image.dtype}")
        print(f"Value Range: [{raw_image.min()}, {raw_image.max()}]")

        return raw_image, random_idx

# --- EXECUTION ---
h5_file_path = "your_data_file.h5" # Enter your .h5 path here
raw_data, idx = extract_raw_h5_image(path)

# --- SAVE AS RAW NUMPY FILE ---
# This saves the data with ZERO changes to values or shape
np.save(f"raw_image_{idx}.npy", raw_data)
print(f"Successfully saved as raw_image_{idx}.npy")


Extracted Image Index: 3459
Raw Shape: (125, 125, 8)
Raw Data Type: float32
Value Range: [0.0, 255.0]
Successfully saved as raw_image_3459.npy
